# LPU Campus Parking Intelligence System
## Notebook 01: Comprehensive Exploratory Data Analysis & Data Quality Assessment
**Academic Evaluation Component**: Rubric #2 (4 Marks) — Dataset, EDA & Data Quality Analysis  
**Team Project**: Intelligent Parking Occupancy Prediction & Navigation Assistant for Lovely Professional University  

---
### 1. Project Background & Real-World Problem Definition
Lovely Professional University (LPU) encompasses over 600 acres with 12 distinct parking zones, handling tens of thousands of faculty, student, and visitor vehicles daily. Drivers frequently encounter sudden saturation during class transitions and exam periods, resulting in traffic congestion and wasted commute time.

**Machine Learning Objective**:
Predict real-time parking occupancy counts across multi-horizons ($t+15$, $t+30$, $t+60$ minutes) and estimate the probability of zone saturation ($P(\text{Occ} \ge 90\%)$) to route drivers proactively before congestion cascades.

This notebook executes an in-depth data quality audit, stationarity assessment, autocorrelation structure analysis, academic calendar surges, and feature correlation mapping.

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

### 2. Dataset Acquisition & Structural Overview
We ingest the continuous 5-minute time series recorded across all 12 LPU campus parking zones.

In [ ]:
data_path = "../data/processed/lpu_timeseries.csv"
if not os.path.exists(data_path):
    data_path = "data/processed/lpu_timeseries.csv"

df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Total Observations: {len(df):,}")
print(f"Temporal Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Monitored Campus Zones: {df['zone_id'].nunique()}")
display(df.head())

### 3. Data Quality, Null Values & Sensor Fault Diagnostics
Verifying sensor completeness, missing intervals, and simulated camera sensor outage dropouts.

In [ ]:
print("=== Missing Value Audit ===")
print(df.isnull().sum())

print("\n=== Camera Status Distribution ===")
display(df['camera_status'].value_counts())

offline_count = (df['camera_status'] == 'offline').sum()
print(f"Total Offline Ingestion Steps: {offline_count} ({offline_count / len(df) * 100:.2f}%)")

### 4. Campus Parking Zone Capacity & Descriptive Statistics
Assessing operational capacities and vehicle count distributions across all 12 campus parking hubs.

In [ ]:
zone_stats = df.groupby('zone_id').agg(
    Operational_Capacity=('operational_capacity', 'first'),
    Mean_Vehicles=('vehicle_count', 'mean'),
    Std_Vehicles=('vehicle_count', 'std'),
    Max_Vehicles=('vehicle_count', 'max'),
    Min_Vehicles=('vehicle_count', 'min')
).reset_index()

zone_stats['Utilization_Rate_%'] = (zone_stats['Mean_Vehicles'] / zone_stats['Operational_Capacity']) * 100
display(zone_stats.sort_values(by='Operational_Capacity', ascending=False))

### 5. Time Series Visual Inspection: Diurnal & Weekly Cycles
Examining vehicle inflow and outflow patterns across representative zones (e.g. Block 1 Academic Hub vs BH2 Hostel Zone).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for zone in ['ZONE_BLOCK1', 'ZONE_BH2', 'ZONE_MBFS']:
    zone_slice = df[df['zone_id'] == zone].iloc[:288 * 3] # 3 days of 5-min intervals
    ax.plot(zone_slice['timestamp'], zone_slice['vehicle_count'], label=zone, alpha=0.85)

ax.set_title("3-Day Continuous Vehicle Count Trace across Campus Archetypes", fontweight='bold')
ax.set_xlabel("Timestamp")
ax.set_ylabel("Occupancy (Vehicles)")
ax.legend()
plt.tight_layout()
plt.show()

### 6. Stationarity Analysis: Augmented Dickey-Fuller (ADF) Test
Evaluating unit roots on the continuous series to determine differencing requirements for autoregressive modelling.

In [ ]:
def evaluate_stationarity(series, zone_name):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"ADF Test for {zone_name}:")
    print(f"  Test Statistic: {result[0]:.4f}")
    print(f"  p-value:        {result[1]:.4e}")
    print(f"  Critical Values: 1%: {result[4]['1%']:.2f}, 5%: {result[4]['5%']:.2f}")
    if result[1] < 0.05:
        print("  => Reject H0: Series is stationary at 95% confidence.")
    else:
        print("  => Fail to reject H0: Non-stationary unit root present.")

block1_counts = df[df['zone_id'] == 'ZONE_BLOCK1']['vehicle_count']
evaluate_stationarity(block1_counts, "ZONE_BLOCK1 Raw Occupancy")

# Differenced series (velocity)
evaluate_stationarity(block1_counts.diff(3).dropna(), "ZONE_BLOCK1 Differenced (Δ15m)")

### 7. Autocorrelation (ACF) & Partial Autocorrelation (PACF) Structure
Justifying the selection of autoregressive lag features ($t-15, t-30, t-45, t-60$ minutes).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(block1_counts.dropna(), lags=36, ax=axes[0], title="Autocorrelation (ACF) - Up to 3 Hours")
plot_pacf(block1_counts.dropna(), lags=36, ax=axes[1], title="Partial Autocorrelation (PACF)")
axes[0].set_xlabel("Lag Steps (5-min intervals)")
axes[1].set_xlabel("Lag Steps (5-min intervals)")
plt.tight_layout()
plt.show()

### 8. Academic Bell Surges & Class Transition Windows
Analyzing occupancy spike velocities during the 5 scheduled LPU class transitions (8:45, 10:45, 12:45, 14:45, 16:45).

In [ ]:
surge_data = df[df['zone_id'] == 'ZONE_BLOCK1']
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x='is_class_transition', y='vehicle_count', data=surge_data, palette=['#94a3b8', '#ea580c'], ax=ax)
ax.set_xticklabels(['Regular Interval', 'Class Transition Bell Window (±15 min)'])
ax.set_title("Occupancy Distribution During Class Transition Surges (Block 1)", fontweight='bold')
ax.set_ylabel("Occupancy Count")
plt.tight_layout()
plt.show()

### 9. Academic Calendar Shift: Mid-Term Examination Periods
Comparing regular instructional days against mid-term examination week inflow.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(data=df[df['is_exam_day'] == 0]['vehicle_count'], label='Regular Semester Day', shade=True, color='#3b82f6')
sns.kdeplot(data=df[df['is_exam_day'] == 1]['vehicle_count'], label='Mid-Term Exam Day', shade=True, color='#ef4444')
ax.set_title("Occupancy Density Shift: Regular Days vs. Exam Session Periods", fontweight='bold')
ax.set_xlabel("Occupancy (Vehicles)")
ax.legend()
plt.tight_layout()
plt.show()

### 10. Saturation Target & Class Imbalance Analysis ($P(\text{Occ} \ge 90\%)$)
Investigating the extreme class imbalance in parking saturation events.

In [ ]:
df['occupancy_ratio'] = df['vehicle_count'] / df['operational_capacity']
df['is_saturated'] = (df['occupancy_ratio'] >= 0.90).astype(int)

sat_counts = df['is_saturated'].value_counts()
sat_pct = (sat_counts / len(df)) * 100
print("=== Saturation Target Imbalance (Threshold >= 90% Capacity) ===")
print(f"Non-Saturated [0]: {sat_counts[0]:,} ({sat_pct[0]:.2f}%)")
print(f"Saturated [1]:     {sat_counts[1]:,} ({sat_pct[1]:.2f}%)")
print(f"Imbalance Ratio:    1 : {sat_counts[0] / sat_counts[1]:.1f}")

### 11. Feature Correlation Matrix & Multicollinearity Assessment
Generating the Pearson correlation matrix across engineered lag features, rolling windows, and target variables.

In [ ]:
from features.feature_pipeline import TimeSeriesFeaturePipeline

pipe = TimeSeriesFeaturePipeline()
trans = pipe.transform_zone_series(df)
X, targets = pipe.prepare_training_matrices(trans)
X_corr = X.copy()
X_corr['Target_H15'] = targets['H15']
X_corr['Target_SAT15'] = targets['SAT15']

plt.figure(figsize=(13, 10))
sns.heatmap(X_corr.corr(), cmap='coolwarm', annot=False, linewidths=0.5)
plt.title("LPU Campus Parking Feature Correlation Matrix", fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
os.makedirs("../artifacts/figures", exist_ok=True)
plt.savefig("../artifacts/figures/feature_correlation_heatmap.png", bbox_inches='tight')
plt.show()

### 12. EDA Synthesis & Modeling Strategy
1. **Strong Autoregressive Dependency**: PACF cuts off after lag 1-3, confirming that lags ($t-15, t-30, t-45, t-60$) carry high predictive power.
2. **Inflection Fragility in Baselines**: Simple persistence baselines experience high error spikes during bell transition surges.
3. **Severe Saturation Imbalance**: Less than 10% of time steps reach $\ge 90\%$ occupancy, necessitating Precision-Recall AUC (PR-AUC) and tuned classification thresholds over standard Accuracy.
4. **Chronological Integrity**: Standard K-Fold CV causes massive temporal lookahead leakage; strict chronological train/val/test splitting with an enforced safety purge gap is essential.